# 85) Colab Adapter Training (T4)

This notebook trains a LoRA adapter on a free Colab T4 GPU using a self-contained training loop.
It minimizes reliance on repo scripts and uses the calibrated T4 config defaults.

Optional: mount Drive for persistence; otherwise artifacts are stored under `/content`.

After download, place `lora_adapter` into `artifacts/runs/finetune_unsloth/lora_adapter` in your local repo.

In [1]:
# 0) Optional: mount Google Drive for persistence
import importlib.util
import os

DRIVE_MOUNTED = False
is_kaggle = bool(os.getenv("KAGGLE_URL_BASE")) or os.path.exists("/kaggle")
is_colab = importlib.util.find_spec("google.colab") is not None

if is_colab and not is_kaggle:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("Drive mount unsupported in this runtime; continuing without Drive.")
else:
    print("Drive mount skipped (Kaggle or non-Colab runtime).")

Drive mount skipped (Kaggle or non-Colab runtime).


In [1]:
# 1) Confirm GPU runtime (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

Sun Apr 12 11:07:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 1.2) Clear stale GPU processes (skip current kernel)
import os
import subprocess

current_pid = os.getpid()
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
pids = [int(line.strip()) for line in result.stdout.splitlines() if line.strip().isdigit()]
stale = [pid for pid in pids if pid != current_pid]

if not pids:
    print("No GPU processes found.")
elif not stale:
    print(f"GPU process matches current kernel pid: {current_pid}. Nothing to kill.")
else:
    for pid in stale:
        try:
            os.kill(pid, 9)
            print(f"Killed GPU process pid={pid}")
        except Exception as exc:
            print(f"Failed to kill pid={pid}: {exc}")

No GPU processes found.


In [4]:
# 1.5) Optional: set HF_TOKEN for gated models
import os
import getpass

if not os.getenv("HF_TOKEN"):
    token = getpass.getpass("HF token (press Enter to skip): ")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN set for this session.")
    else:
        print("No token set.")
else:
    print("HF_TOKEN already set in environment.")

HF_TOKEN set for this session.


In [ ]:
# 2) Clone repo + move into it
from pathlib import Path
import os

REPO_URL = "https://github.com/aaliyan1230/polyglot-grounded-qa.git"
IS_KAGGLE = bool(os.getenv("KAGGLE_URL_BASE")) or Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_ROOT = WORK_ROOT / "polyglot-grounded-qa"
if not REPO_ROOT.exists():
    !git clone {REPO_URL} {REPO_ROOT}

%cd {REPO_ROOT}
!git pull --ff-only
print("WORK_ROOT:", WORK_ROOT)
print("REPO_ROOT:", REPO_ROOT)

/content/polyglot-grounded-qa
Already up to date.


In [ ]:
# 3) Configure export paths (Drive optional)
from pathlib import Path
import os

IS_KAGGLE = bool(os.getenv("KAGGLE_URL_BASE")) or Path("/kaggle/working").exists()
WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working") if IS_KAGGLE else Path("/content"))
REPO_ROOT = REPO_ROOT if "REPO_ROOT" in globals() else WORK_ROOT / "polyglot-grounded-qa"
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"

drive_ready = bool(globals().get("DRIVE_MOUNTED", False)) and DRIVE_MYDRIVE.exists()
if drive_ready:
    DRIVE_ROOT = DRIVE_MYDRIVE / "polyglot_grounded_qa"
else:
    DRIVE_ROOT = WORK_ROOT / "polyglot_grounded_qa_exports"

DRIVE_RUN_DIR = DRIVE_ROOT / "finetune_unsloth"
DRIVE_ARTIFACTS_DIR = DRIVE_ROOT / "exports"

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT:", WORK_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("DRIVE_RUN_DIR:", DRIVE_RUN_DIR)
print("DRIVE_ARTIFACTS_DIR:", DRIVE_ARTIFACTS_DIR)

REPO_ROOT: /content/polyglot-grounded-qa
DRIVE_ROOT: /content/polyglot_grounded_qa_exports
DRIVE_RUN_DIR: /content/polyglot_grounded_qa_exports/finetune_unsloth
DRIVE_ARTIFACTS_DIR: /content/polyglot_grounded_qa_exports/exports


In [6]:
# 4) Install stable training dependencies (single clean set)
import os
import sys

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["DISABLE_TRANSFORMERS_AV"] = "1"

print("Python:", sys.version)
print("Executable:", sys.executable)

%pip install -U pip setuptools wheel packaging
%pip uninstall -y torchvision torchaudio tensorflow keras opencv-python opencv-python-headless opencv-contrib-python
%pip install -U pyyaml polars duckdb
%pip install -U "transformers==4.46.3" "trl==0.12.2" "peft==0.14.0" "datasets==2.21.0" "accelerate==0.34.2" bitsandbytes
%pip install -U --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.11.4" "pandas==2.2.3" "pyarrow==16.1.0"

import importlib
for m in ["numpy", "scipy", "pandas", "pyarrow", "transformers", "trl", "peft", "datasets", "accelerate", "torch", "bitsandbytes"]:
    try:
        mod = importlib.import_module(m)
        print(m, getattr(mod, "__version__", ""))
    except Exception as exc:
        print(f"WARN: failed to import {m}: {exc}")

print("Dependency installation complete.")
print("If imports fail here, restart runtime once and rerun this cell.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Executable: /usr/bin/python3
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 32.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 151.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 275.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 252.0 MB/s  0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.1.post1
    Uninstalling pytz-2026.1.post1:
      Successfully uninstalled pytz-2026.1.post1
  Attempting uninstall: tzdata━━━━━━━━━━━━━━━━━━ 0/8 [pytz]
    Found existing installation: tzdata 2026.10m 0/8 [pytz]
    Uninstalling tzdata-2026.1:━━

In [7]:
# 5) Ensure formatted SFT files (self-contained)
from pathlib import Path
import json
import math
import random
import shutil
from collections import defaultdict

REPO_ROOT = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
data_dir = REPO_ROOT / "data" / "benchmarks" / "finetune"
formatted_dir = data_dir / "formatted"

train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"
test_path = data_dir / "test.jsonl"

formatted_train = formatted_dir / "train.text.jsonl"
formatted_val = formatted_dir / "val.text.jsonl"
formatted_test = formatted_dir / "test.text.jsonl"

def _read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=True) + "\n")

def _split_bucket(rows: list[dict], train_ratio: float, val_ratio: float):
    n = len(rows)
    train_n = round(n * train_ratio)
    val_n = round(n * val_ratio)
    if n >= 4 and val_n == 0:
        val_n = 1
    test_n = n - train_n - val_n
    if n >= 4 and test_n == 0:
        test_n = 1
        train_n = max(0, n - val_n - test_n)
    train_n = max(0, min(train_n, n))
    val_n = max(0, min(val_n, n - train_n))
    return rows[:train_n], rows[train_n : train_n + val_n], rows[train_n + val_n :]

def _abstain_ratio(rows: list[dict]) -> float:
    if not rows:
        return 0.0
    abstained = sum(1 for row in rows if bool(row["target"].get("abstained", False)))
    return abstained / len(rows)

def _rebalance_abstain_ratio(donor: list[dict], recipient: list[dict], min_ratio: float) -> None:
    if not recipient:
        return
    recipient_abstained = sum(1 for row in recipient if bool(row["target"].get("abstained", False)))
    needed = math.ceil(min_ratio * len(recipient)) - recipient_abstained
    if needed <= 0:
        return
    donor_abstained_idx = [
        idx for idx, row in enumerate(donor) if bool(row["target"].get("abstained", False))
    ]
    recipient_non_abstained_idx = [
        idx for idx, row in enumerate(recipient) if not bool(row["target"].get("abstained", False))
    ]
    swaps = min(needed, len(donor_abstained_idx), len(recipient_non_abstained_idx))
    for i in range(swaps):
        d_idx = donor_abstained_idx[i]
        r_idx = recipient_non_abstained_idx[i]
        donor[d_idx], recipient[r_idx] = recipient[r_idx], donor[d_idx]

# Try to sync from Drive if repo data files are missing.
drive_data_dir = None
if "DRIVE_ROOT" in globals():
    drive_data_dir = Path(DRIVE_ROOT) / "data" / "benchmarks" / "finetune"
    if drive_data_dir.exists():
        for name in [
            "train.jsonl",
            "val.jsonl",
            "test.jsonl",
            "sft_dataset_merged.jsonl",
            "sft_dataset.jsonl",
            "public_sft_dataset.jsonl",
        ]:
            src = drive_data_dir / name
            dst = data_dir / name
            if src.exists() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                print("Copied from Drive:", src, "->", dst)

source_candidates = [
    data_dir / "sft_dataset_merged.jsonl",
    data_dir / "sft_dataset.jsonl",
    data_dir / "public_sft_dataset.jsonl",
]
source_path = next((p for p in source_candidates if p.exists()), None)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    if source_path is None:
        print("No dataset found; generating a minimal seed dataset for quick training.")
        seed_chunks = [
            {
                "doc_id": "seed-doc-1",
                "chunk_id": "seed-chunk-1",
                "text": "Polyglot grounded QA uses retrieval with citations and verification.",
            },
            {
                "doc_id": "seed-doc-2",
                "chunk_id": "seed-chunk-2",
                "text": "Language packs inherit base behavior and only override locale-specific rules.",
            },
        ]
        queries = [
            "What is grounded QA?",
            "How does language-pack inheritance work?",
            "Why are citations required?",
        ]
        answer_map = {
            "What is grounded QA?": ("Polyglot grounded QA uses retrieval with citations and verification.", ["seed-chunk-1"]),
            "How does language-pack inheritance work?": ("Language packs inherit base behavior and only override locale-specific rules.", ["seed-chunk-2"]),
            "Why are citations required?": ("Grounded QA answers with explicit evidence citations.", ["seed-chunk-1"]),
        }
        languages = ["base", "es", "es-MX", "fr", "tr"]
        rows: list[dict] = []
        counter = 0
        for language in languages:
            for query in queries:
                answer, citations = answer_map[query]
                rows.append({
                    "id": f"sft-{counter:06d}",
                    "language": language,
                    "query": query,
                    "retrieved_chunks": seed_chunks,
                    "target": {
                        "answer": answer,
                        "citations": citations,
                        "abstained": False,
                        "reason": "",
                    },
                    "label_type": "answerable",
                    "source": "seed-minimal",
                })
                counter += 1
                rows.append({
                    "id": f"sft-{counter:06d}",
                    "language": language,
                    "query": f"{query} [insufficient-evidence]",
                    "retrieved_chunks": [],
                    "target": {
                        "answer": "I do not have enough evidence.",
                        "citations": [],
                        "abstained": True,
                        "reason": "insufficient_evidence",
                    },
                    "label_type": "insufficient_evidence",
                    "source": "seed-minimal",
                })
                counter += 1
        source_path = data_dir / "sft_dataset.jsonl"
        _write_jsonl(source_path, rows)
        print("Wrote seed dataset to", source_path)
    rows = _read_jsonl(source_path)
    if not rows:
        raise ValueError(f"{source_path} is empty.")
    rng = random.Random(42)
    by_language: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        language = str(row.get("language", "base"))
        by_language[language].append(row)
    train_rows: list[dict] = []
    val_rows: list[dict] = []
    test_rows: list[dict] = []
    for _, bucket in by_language.items():
        rng.shuffle(bucket)
        train, val, test = _split_bucket(bucket, train_ratio=0.8, val_ratio=0.1)
        train_rows.extend(train)
        val_rows.extend(val)
        test_rows.extend(test)
    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    rng.shuffle(test_rows)
    _rebalance_abstain_ratio(train_rows, val_rows, min_ratio=0.2)
    _rebalance_abstain_ratio(train_rows, test_rows, min_ratio=0.2)
    _write_jsonl(train_path, train_rows)
    _write_jsonl(val_path, val_rows)
    _write_jsonl(test_path, test_rows)
    print("Wrote splits from", source_path)

def _retrieval_block(chunks: list[dict]) -> str:
    if not chunks:
        return "[NO_RETRIEVED_EVIDENCE]"
    lines = []
    for chunk in chunks:
        chunk_id = str(chunk.get("chunk_id", ""))
        text = str(chunk.get("text", "")).strip()
        lines.append(f"[{chunk_id}] {text}")
    return "\n".join(lines)

def _target_json(row: dict) -> str:
    target = row.get("target", {})
    payload = {
        "answer": target.get("answer", ""),
        "citations": target.get("citations", []),
        "abstained": bool(target.get("abstained", False)),
        "reason": target.get("reason", ""),
    }
    return json.dumps(payload, ensure_ascii=True)

def _instruction(row: dict) -> str:
    language = str(row.get("language", "base"))
    query = str(row.get("query", "")).strip()
    evidence = _retrieval_block(row.get("retrieved_chunks", []))
    return (
        "You are a grounded QA model. Use only evidence below.\n"
        "If evidence is insufficient, abstain.\n"
        "Return strict JSON with keys: answer, citations, abstained, reason.\n\n"
        f"Language: {language}\n"
        f"Query: {query}\n"
        f"Evidence:\n{evidence}"
    )

def _build_text_record(row: dict) -> dict:
    prompt = _instruction(row)
    completion = _target_json(row)
    return {
        "id": row.get("id", ""),
        "language": row.get("language", "base"),
        "text": f"<|user|>\n{prompt}\n<|assistant|>\n{completion}",
        "source": row.get("source", ""),
        "label_type": row.get("label_type", ""),
    }

if formatted_train.exists() and formatted_val.exists() and formatted_test.exists():
    print("Formatted files already exist.")
else:
    for split, path in [("train", train_path), ("val", val_path), ("test", test_path)]:
        rows = _read_jsonl(path)
        if not rows:
            raise ValueError(f"{path} is empty.")
        out = [_build_text_record(row) for row in rows]
        _write_jsonl(formatted_dir / f"{split}.text.jsonl", out)
    print("Wrote formatted text files to", formatted_dir)

print("Data prep OK.")
print("Train:", formatted_train.exists(), "Val:", formatted_val.exists(), "Test:", formatted_test.exists())

Formatted files already exist.
Data prep OK.
Train: True Val: True Test: True


In [8]:
# 6) Load T4 config and apply optional overrides
from pathlib import Path
import yaml

REPO_ROOT = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
cfg_path = REPO_ROOT / "configs" / "finetune" / "cloud_unsloth_qlora.yaml"
CFG = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 1024
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8
MAX_STEPS = 300
SAVE_STEPS = 100
LOGGING_STEPS = 20
EVAL_STRATEGY = "no"
RUN_NAME = "colab-t4-qwen2.5-3b-qlora"

model_cfg = CFG.get("model", {})
model_cfg["name"] = MODEL_NAME
CFG["model"] = model_cfg

train_cfg = CFG.get("training", {})
train_cfg["max_seq_length"] = int(MAX_SEQ_LENGTH)
train_cfg["per_device_train_batch_size"] = int(PER_DEVICE_BATCH)
train_cfg["gradient_accumulation_steps"] = int(GRAD_ACCUM)
train_cfg["max_steps"] = int(MAX_STEPS)
train_cfg["save_steps"] = int(SAVE_STEPS)
train_cfg["logging_steps"] = int(LOGGING_STEPS)
train_cfg["eval_strategy"] = EVAL_STRATEGY
train_cfg["eval_steps"] = int(SAVE_STEPS)
CFG["training"] = train_cfg
CFG["run_name"] = RUN_NAME

print("Loaded", cfg_path)
print("Run name:", RUN_NAME)
print("Model:", MODEL_NAME)
print("Training config:", train_cfg)

Loaded /content/polyglot-grounded-qa/configs/finetune/cloud_unsloth_qlora.yaml
Run name: colab-t4-qwen2.5-3b-qlora
Model: Qwen/Qwen2.5-3B-Instruct
Training config: {'max_seq_length': 1024, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 8, 'learning_rate': 0.0002, 'warmup_steps': 20, 'max_steps': 300, 'logging_steps': 20, 'save_steps': 100, 'optim': 'adamw_8bit', 'seed': 42, 'eval_strategy': 'no', 'eval_steps': 100}


In [ ]:
# 7) Train adapter (output persisted on Drive if mounted)
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["DISABLE_TRANSFORMERS_AV"] = "1"
os.environ["WANDB_DISABLED"] = "true"

import importlib
from pathlib import Path
import yaml

project_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
if "CFG" in globals():
    cfg = CFG
else:
    cfg_path = project_root / "configs" / "finetune" / "cloud_unsloth_qlora.yaml"
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

model_cfg = cfg.get("model", {})
lora_cfg = cfg.get("lora", {})
train_cfg = cfg.get("training", {})
data_cfg = cfg.get("data", {})

model_name = str(model_cfg.get("name", "Qwen/Qwen2.5-3B-Instruct"))
max_seq_length = int(train_cfg.get("max_seq_length", 1024))

train_file = project_root / str(
    data_cfg.get("train", "data/benchmarks/finetune/formatted/train.text.jsonl")
 )
val_file = project_root / str(
    data_cfg.get("val", "data/benchmarks/finetune/formatted/val.text.jsonl")
 )

if "DRIVE_RUN_DIR" in globals():
    OUTPUT_DIR = Path(DRIVE_RUN_DIR)
else:
    OUTPUT_DIR = Path("/content/polyglot_grounded_qa_exports/finetune_unsloth")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets_module = importlib.import_module("datasets")
trl_module = importlib.import_module("trl")
transformers_module = importlib.import_module("transformers")
peft_module = importlib.import_module("peft")
torch_module = importlib.import_module("torch")

load_dataset = datasets_module.load_dataset
SFTConfig = trl_module.SFTConfig
SFTTrainer = trl_module.SFTTrainer
AutoModelForCausalLM = transformers_module.AutoModelForCausalLM
AutoTokenizer = transformers_module.AutoTokenizer
BitsAndBytesConfig = getattr(transformers_module, "BitsAndBytesConfig", None)
LoraConfig = peft_module.LoraConfig
get_peft_model = peft_module.get_peft_model

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {"device_map": "auto", "torch_dtype": torch_module.float16}
if BitsAndBytesConfig is not None:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
peft_cfg = LoraConfig(
    r=int(lora_cfg.get("rank", 16)),
    lora_alpha=int(lora_cfg.get("alpha", 16)),
    lora_dropout=float(lora_cfg.get("dropout", 0.0)),
    target_modules=list(
        lora_cfg.get(
            "target_modules",
            ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        )
    ),
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_cfg)

train_ds = load_dataset("json", data_files=str(train_file), split="train")
val_ds = load_dataset("json", data_files=str(val_file), split="train")

eval_strategy = str(train_cfg.get("eval_strategy", "steps"))
eval_steps = int(train_cfg.get("eval_steps", train_cfg.get("save_steps", 50)))

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_seq_length=max_seq_length,
    per_device_train_batch_size=int(train_cfg.get("per_device_train_batch_size", 2)),
    gradient_accumulation_steps=int(train_cfg.get("gradient_accumulation_steps", 8)),
    learning_rate=float(train_cfg.get("learning_rate", 2e-4)),
    warmup_steps=int(train_cfg.get("warmup_steps", 20)),
    max_steps=int(train_cfg.get("max_steps", 300)),
    logging_steps=int(train_cfg.get("logging_steps", 20)),
    save_steps=int(train_cfg.get("save_steps", 100)),
    optim=str(train_cfg.get("optim", "adamw_8bit")),
    seed=int(train_cfg.get("seed", 42)),
    eval_strategy=eval_strategy,
    eval_steps=eval_steps,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_args,
)

trainer.train()
adapter_dir = OUTPUT_DIR / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Saved adapter to {adapter_dir}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aaliyan1230 (aaliyan12300) to https://api.wandb.

Step,Training Loss
20,4.573000
40,0.154600
60,0.046400
80,0.043100
100,0.042600
120,0.041900
140,0.042400
160,0.041600
180,0.041100
200,0.040900


Saved adapter to /content/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter


In [ ]:
# 8) Verify + package adapter and expose Kaggle-downloadable links
from pathlib import Path
import os
import shutil
from datetime import UTC, datetime
from IPython.display import HTML, display

if "OUTPUT_DIR" not in globals():
    raise RuntimeError("Run the training cell first to define OUTPUT_DIR.")

IS_KAGGLE = bool(os.getenv("KAGGLE_URL_BASE")) or Path("/kaggle/working").exists()
WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working") if IS_KAGGLE else Path("/content"))
adapter_dir = Path(OUTPUT_DIR) / "lora_adapter"
if not adapter_dir.exists():
    raise FileNotFoundError(f"Missing adapter directory: {adapter_dir}")

ts = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
archive_base = Path(WORK_ROOT) / f"lora_adapter_{ts}"
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=adapter_dir))

drive_zip = None
if "DRIVE_ARTIFACTS_DIR" in globals():
    drive_zip = Path(DRIVE_ARTIFACTS_DIR) / zip_path.name
    if drive_zip != zip_path:
        shutil.copy2(zip_path, drive_zip)

kaggle_working_zip = Path("/kaggle/working") / zip_path.name
kaggle_working_zip.parent.mkdir(parents=True, exist_ok=True)
if kaggle_working_zip != zip_path:
    shutil.copy2(zip_path, kaggle_working_zip)

# Also place a copy in the current notebook working directory for /files/<name> downloads.
cwd_zip = Path.cwd() / zip_path.name
if cwd_zip != kaggle_working_zip:
    shutil.copy2(kaggle_working_zip, cwd_zip)

size_mb = kaggle_working_zip.stat().st_size / (1024 * 1024)
print("Adapter dir:", adapter_dir)
print("Local zip:", zip_path)
if drive_zip:
    print("Drive zip:", drive_zip)
print("Kaggle working zip:", kaggle_working_zip)
print(f"Zip size: {size_mb:.2f} MB")

download_name = kaggle_working_zip.name
display(
    HTML(
        f"""
<div>
  <p><b>Try these download links (in order):</b></p>
  <ol>
    <li><a href='/files/{download_name}' target='_blank' download>{download_name} via /files route</a></li>
    <li><a href='{download_name}' target='_blank' download>{download_name} relative link</a></li>
    <li><a href='/kaggle/working/{download_name}' target='_blank' download>{download_name} direct working path</a></li>
  </ol>
  <p>If all links are blocked, click <b>Save Version</b> and download from that version's <b>Output files</b>.</p>
</div>
"""
    )
)

Adapter dir: /content/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter
Local zip: /content/lora_adapter_20260403_182212.zip
Drive zip: /content/polyglot_grounded_qa_exports/exports/lora_adapter_20260403_182212.zip
Kaggle working zip: /kaggle/working/lora_adapter_20260403_182212.zip
Zip size: 108.73 MB


In [15]:
# 9) Publish adapter directly as a Kaggle dataset artifact
from pathlib import Path
import json
import shutil
import subprocess
from datetime import UTC, datetime

DATASET_ID = "aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts"
STAGE_ROOT = Path("/kaggle/working/kaggle_artifact_stage")
ADAPTER_SRC = Path(OUTPUT_DIR) / "lora_adapter" if "OUTPUT_DIR" in globals() else None

if ADAPTER_SRC is None or not ADAPTER_SRC.exists():
    raise FileNotFoundError("Adapter directory missing. Run Cell 11 (training) first.")

adapter_dst = STAGE_ROOT / "artifacts" / "runs" / "finetune_unsloth" / "lora_adapter"
if adapter_dst.exists():
    shutil.rmtree(adapter_dst)
adapter_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(ADAPTER_SRC, adapter_dst)

# Keep the latest zip in artifacts/exports for easier browsing.
zip_candidates = sorted(Path("/kaggle/working").glob("lora_adapter_*.zip"))
if zip_candidates:
    exports_dir = STAGE_ROOT / "artifacts" / "exports"
    exports_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_candidates[-1], exports_dir / zip_candidates[-1].name)

metadata = {
    "title": "polyglot-grounded-qa-finetune-artifacts",
    "id": DATASET_ID,
    "licenses": [{"name": "Apache-2.0"}],
}
metadata_path = STAGE_ROOT / "dataset-metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print("Running:", " ".join(cmd))
    cp = subprocess.run(cmd, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    return cp

# Try version first if dataset exists, else create it.
status_cp = run_cmd(["kaggle", "datasets", "status", DATASET_ID])
message = f"sync lora adapter {datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC')}"

if status_cp.returncode == 0:
    publish_cp = run_cmd([
        "kaggle", "datasets", "version",
        "-p", str(STAGE_ROOT),
        "-m", message,
        "--dir-mode", "zip",
    ])
else:
    publish_cp = run_cmd([
        "kaggle", "datasets", "create",
        "-p", str(STAGE_ROOT),
        "--dir-mode", "zip",
    ])

if publish_cp.returncode != 0:
    raise RuntimeError(
        "Kaggle publish failed. If credentials are missing, set KAGGLE_USERNAME and KAGGLE_KEY "
        "in Kaggle notebook Secrets (do not paste them in chat)."
    )

print("Published dataset artifact:", DATASET_ID)
print("Staged adapter path:", adapter_dst)
print("Stage root:", STAGE_ROOT)

Running: kaggle datasets status aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
ready
Running: kaggle datasets version -p /kaggle/working/kaggle_artifact_stage -m sync lora adapter 2026-04-03 18:25:35 UTC --dir-mode zip
Starting upload for file artifacts.zip
Upload successful: artifacts.zip (217MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
0%|          | 0.00/217M [00:00<?, ?B/s]
  5%|▍         | 10.1M/217M [00:00<00:02, 102MB/s]
 13%|█▎        | 28.8M/217M [00:00<00:01, 149MB/s]
 20%|█▉        | 43.0M/217M [00:00<00:01, 97.5MB/s]
 29%|██▉       | 62.8M/217M [00:00<00:01, 118MB/s] 
 35%|███▍      | 75.1M/217M [00:00<00:01, 114MB/s]
 40%|███▉      | 86.9M/217M [00:00<00:01, 110MB/s]
 45%|████▌     | 98.9M/217M [00:00<00:01, 114MB/s]
 51%|█████     | 110M/217M [00:01<00:01, 105MB/s] 
 59%|█████▊    | 128M/217M [00:01<00:00, 108MB/s]
 63%|██████▎   | 138M/217M [00:01<00:00, 100MB/s

In [23]:
# 10) Run trained-adapter eval append on Kaggle runtime
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
repo_adapter_dir = repo_root / "artifacts" / "runs" / "finetune_unsloth" / "lora_adapter"
raw_pred = repo_root / "artifacts" / "runs" / "raw_model_predictions_adapter.jsonl"
norm_pred = repo_root / "artifacts" / "runs" / "tuned_predictions_adapter.jsonl"

candidate_dirs = []
if "OUTPUT_DIR" in globals():
    candidate_dirs.append(Path(OUTPUT_DIR) / "lora_adapter")
candidate_dirs.append(Path("/content/polyglot_grounded_qa_exports/finetune_unsloth/lora_adapter"))

adapter_src = next((p for p in candidate_dirs if p.exists()), None)
if adapter_src is None:
    raise FileNotFoundError("Could not find adapter dir in OUTPUT_DIR or /content/polyglot_grounded_qa_exports.")

if repo_adapter_dir.exists():
    shutil.rmtree(repo_adapter_dir)
repo_adapter_dir.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(adapter_src, repo_adapter_dir)
print("Synced adapter into repo path:", repo_adapter_dir)

def run_repo_cmd(cmd: list[str]) -> None:
    print("Running:", " ".join(cmd))
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    src_path = str(repo_root / "src")
    env["PYTHONPATH"] = src_path + os.pathsep + env.get("PYTHONPATH", "")
    cp = subprocess.run(cmd, cwd=repo_root, env=env, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    if cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {' '.join(cmd)}")

py = sys.executable
run_repo_cmd([
    py, "scripts/generate_tuned_predictions.py",
    "--mode", "hf-adapter",
    "--test-file", "data/benchmarks/finetune/test.jsonl",
    "--base-model", "Qwen/Qwen2.5-3B-Instruct",
    "--adapter-path", str(repo_adapter_dir),
    "--output", str(raw_pred),
    "--max-new-tokens", "192",
    "--temperature", "0.0",
])
run_repo_cmd([
    py, "scripts/normalize_tuned_predictions.py",
    "--input", str(raw_pred),
    "--output", str(norm_pred),
])
run_repo_cmd([
    py, "scripts/run_finetune_eval.py",
    "--variant", "tuned-adapter-v1",
    "--predictions", str(norm_pred),
    "--test-file", "data/benchmarks/finetune/test.jsonl",
    "--append",
])
print("Kaggle-side tuned-adapter eval append completed.")

Synced adapter into repo path: /content/polyglot-grounded-qa/artifacts/runs/finetune_unsloth/lora_adapter
Running: /usr/bin/python3 scripts/generate_tuned_predictions.py --mode hf-adapter --test-file data/benchmarks/finetune/test.jsonl --base-model Qwen/Qwen2.5-3B-Instruct --adapter-path /content/polyglot-grounded-qa/artifacts/runs/finetune_unsloth/lora_adapter --output /content/polyglot-grounded-qa/artifacts/runs/raw_model_predictions_adapter.jsonl --max-new-tokens 192 --temperature 0.0
Wrote 5 predictions to /content/polyglot-grounded-qa/artifacts/runs/raw_model_predictions_adapter.jsonl
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.04s/it]
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3

In [21]:
# Debug helper: rerun adapter prediction and print only last lines
from pathlib import Path
import os
import subprocess
import sys

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
repo_adapter_dir = repo_root / "artifacts" / "runs" / "finetune_unsloth" / "lora_adapter"
raw_pred = repo_root / "artifacts" / "runs" / "raw_model_predictions_adapter.jsonl"

cmd = [
    sys.executable, "scripts/generate_tuned_predictions.py",
    "--mode", "hf-adapter",
    "--test-file", "data/benchmarks/finetune/test.jsonl",
    "--base-model", "Qwen/Qwen2.5-3B-Instruct",
    "--adapter-path", str(repo_adapter_dir),
    "--output", str(raw_pred),
    "--max-new-tokens", "192",
    "--temperature", "0.0",
]

env = os.environ.copy()
src_path = str(repo_root / "src")
env["PYTHONPATH"] = src_path + os.pathsep + env.get("PYTHONPATH", "")

cp = subprocess.run(cmd, cwd=repo_root, env=env, text=True, capture_output=True)
print("Return code:", cp.returncode)
print("--- STDOUT last 30 lines ---")
for line in cp.stdout.splitlines()[-30:]:
    print(line)
print("--- STDERR last 30 lines ---")
for line in cp.stderr.splitlines()[-30:]:
    print(line)

Return code: 1
--- STDOUT last 30 lines ---
--- STDERR last 30 lines ---
    output = module._old_forward(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/qwen2/modeling_qwen2.py", line 1164, in forward
    outputs = self.model(
              ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/qwen2/modeling_qwen2.py", line 871, in forward
    position_embeddings = self.rotary_emb(hidden_states, position_ids)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/l

In [24]:
# 11) Publish refreshed full artifacts tree to Kaggle dataset
from pathlib import Path
import json
import shutil
import subprocess
from datetime import UTC, datetime

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path("/content/polyglot-grounded-qa")
dataset_id = "aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts"
stage_root = Path("/kaggle/working/kaggle_artifact_stage_full")
src_artifacts = repo_root / "artifacts"
dst_artifacts = stage_root / "artifacts"

if not src_artifacts.exists():
    raise FileNotFoundError(f"Missing artifacts directory: {src_artifacts}")

if dst_artifacts.exists():
    shutil.rmtree(dst_artifacts)
dst_artifacts.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src_artifacts, dst_artifacts)

metadata = {
    "title": "polyglot-grounded-qa-finetune-artifacts",
    "id": dataset_id,
    "licenses": [{"name": "Apache-2.0"}],
}
(stage_root / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

def run_publish_cmd(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print("Running:", " ".join(cmd))
    cp = subprocess.run(cmd, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.stderr.strip():
        print(cp.stderr.strip())
    return cp

status_cp = run_publish_cmd(["kaggle", "datasets", "status", dataset_id])
message = f"sync full artifacts {datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC')}"

if status_cp.returncode == 0:
    publish_cp = run_publish_cmd([
        "kaggle", "datasets", "version",
        "-p", str(stage_root),
        "-m", message,
        "--dir-mode", "zip",
    ])
else:
    publish_cp = run_publish_cmd([
        "kaggle", "datasets", "create",
        "-p", str(stage_root),
        "--dir-mode", "zip",
    ])

if publish_cp.returncode != 0:
    raise RuntimeError("Failed to publish full artifacts dataset version.")

print("Published refreshed artifacts to:", dataset_id)
print("Staged from:", src_artifacts)

Running: kaggle datasets status aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
ready
Running: kaggle datasets version -p /kaggle/working/kaggle_artifact_stage_full -m sync full artifacts 2026-04-03 18:34:11 UTC --dir-mode zip
Starting upload for file artifacts.zip
Upload successful: artifacts.zip (109MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/aaliyanshaikh/polyglot-grounded-qa-finetune-artifacts
0%|          | 0.00/109M [00:00<?, ?B/s]
  8%|▊         | 8.98M/109M [00:00<00:01, 59.4MB/s]
 28%|██▊       | 30.1M/109M [00:00<00:00, 98.4MB/s]
 36%|███▌      | 39.3M/109M [00:00<00:00, 90.6MB/s]
 46%|████▌     | 49.9M/109M [00:00<00:00, 88.1MB/s]
 53%|█████▎    | 58.2M/109M [00:00<00:00, 70.9MB/s]
 66%|██████▌   | 71.4M/109M [00:00<00:00, 85.1MB/s]
 74%|███████▎  | 80.0M/109M [00:01<00:00, 81.6MB/s]
 88%|████████▊ | 96.1M/109M [00:01<00:00, 91.3MB/s]
 97%|█████████▋| 105M/109M [00:01<00:00, 87.5MB/s] 
100%|██████████| 109M/109M [00:01

## Local follow-up (back in VS Code)

1. Copy adapter directory from Drive or download the zip and place it at `artifacts/runs/finetune_unsloth/lora_adapter`.
2. Run:

```bash
uv run python scripts/run_trained_adapter_eval.py \
  --variant tuned-adapter-v1 \
  --append
```

Defaults for base model and adapter path come from `configs/models/default.yaml` under `finetune`.
3. Re-run comparison and takeaway cells in `notebooks/80_final_results.ipynb`.

### Colab run order (top to bottom)
1. Cell 2: mount Drive (optional)
2. Cell 3: GPU check
3. Cell 4: optional HF token
4. Cell 5: clone/pull repo
5. Cell 6: set export paths
6. Cell 7: install dependencies
7. Cell 8: ensure formatted data
8. Cell 9: load T4 config + overrides
9. Cell 10: train adapter
10. Cell 11: package artifacts